In [5]:
import cv2, os

input_folder = 'raw_videos/goats'
output_folder = 'frames/goats'
os.makedirs(output_folder, exist_ok=True)

for video in os.listdir(input_folder):
    cap = cv2.VideoCapture(os.path.join(input_folder, video))
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % 5 == 0:  # save every 5th frame
            cv2.imwrite(f"{output_folder}/frame_{video[:-4]}_{count}.jpg", frame)
        count += 1
    cap.release()


In [11]:
import os

# specify the folder path
folder_path = "frames/goats"

# list all items in the directory
all_items = os.listdir(folder_path)

# filter only files
files = [f for f in all_items if os.path.isfile(os.path.join(folder_path, f))]

# print the count
print("Number of files:", len(files))


Number of files: 461


In [3]:
!pip install opencv-python
!pip install numpy
!pip install torch torchvision torchaudio
!pip install ultralytics



In [5]:
from ultralytics import YOLO

# Load a pre-trained YOLO model
model = YOLO('yolov8n.pt')  # 'n' stands for nano, a lightweight version


100%|█████████████████████████████████████████████████████████████████████████████| 6.25M/6.25M [00:02<00:00, 2.46MB/s]


In [3]:
import ultralytics

print(ultralytics.__version__)


8.3.129


In [21]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

!yolo train model=yolov8n.pt data=data.yaml epochs=20 imgsz=416 device=cpu name=train1




^C


In [45]:
!yolo detect predict model=runs/detect/train1/weights/best.pt source=C:\Users\Dell\OneDrive\Desktop\newdataset\dogs\VID20251012170152.mp4 show=True


WARNING Environment does not support cv2.imshow() or PIL Image.show()
OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'

Ultralytics 8.3.129  Python-3.12.7 torch-2.7.0+cpu CPU (11th Gen Intel Core(TM) i5-1145G7 2.60GHz)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs

video 1/1 (frame 1/450) C:\Users\Dell\OneDrive\Desktop\newdataset\dogs\VID20251012170152.mp4: 256x416 2 cows, 1 dog, 60.1ms
video 1/1 (frame 2/450) C:\Users\Dell\OneDrive\Desktop\newdataset\dogs\VID20251012170152.mp4: 256x416 2 cows, 1 dog, 31.8ms
video 1/1 (frame 3/450) C:\Users\Dell\OneDrive\Desktop\newdataset\dogs\VID20251012170152.mp4: 256x416 2 cows, 1 dog, 30.9ms
video 1/1 (frame 4/450) C:\

In [3]:
pip install ultralytics filterpy imageio[ffmpeg] opencv-python numpy


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import cv2
import numpy as np
import imageio
from ultralytics import YOLO
from filterpy.kalman import KalmanFilter

# ==============================
# 🧩 PATH SETTINGS
# ==============================
input_video  = r"C:\Users\Dell\animal_tracking_dataset\raw_videos\cows\VID20251012171445.mp4"
output_video = r"C:\Users\Dell\animal_tracking_dataset\output_videos\tracked_cows_trajectory.mp4"
os.makedirs(os.path.dirname(output_video), exist_ok=True)

# ==============================
# 🧠 LOAD YOLO MODEL
# ==============================
model = YOLO("yolov8n.pt")   # or yolov8m.pt for better accuracy

# ==============================
# ⚙️ VIDEO SETUP
# ==============================
cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise FileNotFoundError("❌ Could not open input video!")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
width, height = int(cap.get(3)), int(cap.get(4))
frame_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"🎞 Video info: {width}x{height} @ {fps:.2f} FPS ({frame_total} frames)")

writer = imageio.get_writer(
    output_video, fps=fps, codec="libx264",
    format="ffmpeg", pixelformat="yuv420p", macro_block_size=1
)

# ==============================
# 📈 TRAJECTORY STORAGE
# ==============================
tracks_history = {}  # {track_id: [(x, y), ...]}
filters = {}         # {track_id: KalmanFilter}

def create_kalman():
    kf = KalmanFilter(dim_x=4, dim_z=2)
    dt = 1.0
    kf.F = np.array([[1, 0, dt, 0],
                     [0, 1, 0, dt],
                     [0, 0, 1, 0],
                     [0, 0, 0, 1]])  # state transition
    kf.H = np.array([[1, 0, 0, 0],
                     [0, 1, 0, 0]])  # measurement function
    kf.P *= 1000.0  # covariance
    kf.R *= 10.0
    kf.Q = np.eye(4)
    return kf

# ==============================
# 🚀 MAIN LOOP (Tracking + Prediction)
# ==============================
frame_idx = 0
for result in model.track(
    source=input_video,
    tracker="bytetrack.yaml",     # Official tracker
    conf=0.4,
    iou=0.5,
    imgsz=640,
    persist=True,
    stream=True,
    show=False,
    save=False,
    classes=[15,16,17,18,19,20,21,22,23]
):
    frame = result.plot()
    frame_idx += 1

    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids   = result.boxes.id
        if ids is not None:
            ids = ids.int().cpu().numpy()

            for box, obj_id in zip(boxes, ids):
                x1, y1, x2, y2 = box[:4]
                cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)

                # --- initialize Kalman filter for new ID ---
                if obj_id not in filters:
                    filters[obj_id] = create_kalman()
                    filters[obj_id].x[:2] = np.array([[cx], [cy]])
                    tracks_history[obj_id] = []

                # --- update filter with detection ---
                kf = filters[obj_id]
                kf.predict()
                kf.update([cx, cy])
                pred_x, pred_y = int(kf.x[0]), int(kf.x[1])

                # store trajectory
                tracks_history[obj_id].append((pred_x, pred_y))

                # draw trajectory line
                pts = np.array(tracks_history[obj_id], dtype=np.int32)
                for j in range(1, len(pts)):
                    cv2.line(frame, tuple(pts[j-1]), tuple(pts[j]), (0,255,0), 2)

                # draw prediction point (future estimate)
                future_state = np.dot(kf.F, kf.x)
                fx, fy = int(future_state[0]), int(future_state[1])
                cv2.circle(frame, (fx, fy), 5, (0,0,255), -1)
                cv2.putText(frame, f"ID {obj_id}", (cx, cy-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

    writer.append_data(np.asarray(frame))
    if frame_idx % 25 == 0:
        print(f"Processed {frame_idx}/{frame_total} frames...")

writer.close()
cap.release()
print(f"✅ Tracking + Trajectory prediction complete!\n📁 Saved at: {output_video}")


In [1]:
import os

path = r"animal_tracking_dataset\runs\detect\train1\weights\best.pt"

print(os.path.exists(path))

False
